# 🦿 Project 5.3 — Robotic Arm Undo / Redo
**DSA for Mechatronics · Week 05 — Stacks & Queues**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A teach pendant lets an operator jog a 3-joint robotic arm to a series of waypoints.
Every jog command is recorded; pressing **Undo** reverts to the previous configuration
and pressing **Redo** re-applies it.

Two stacks implement this perfectly:
- **undo_stack** — history of all applied commands
- **redo_stack** — commands that were undone (cleared on new move)

A third **min-stack** (auxiliary stack) tracks the minimum gravitational potential energy
configuration seen so far in O(1) time — useful for safe rest-pose lookup.


## Step 1 — Define the arm state and generate a command sequence

In [ ]:
import math

# ── Personalised parameters ─────────────────────────────────────────
N_JOINTS      = 3
N_COMMANDS    = random.randint(18, 28)
N_UNDOS       = random.randint(3, 7)
N_REDOS       = random.randint(1, min(3, N_UNDOS))
LINK_LENGTHS  = [round(random.uniform(0.3, 0.6), 2) for _ in range(N_JOINTS)]
LINK_MASSES   = [round(random.uniform(1.0, 3.5), 2) for _ in range(N_JOINTS)]
ANGLE_STEP    = random.randint(5, 15)   # degrees per jog

def potential_energy(angles):
    """Simplified gravitational PE: sum m_i * g * h_i  (g=9.81, heights from base)."""
    g = 9.81
    x, y = 0.0, 0.0
    total_pe = 0.0
    for i, (L, m, theta_deg) in enumerate(zip(LINK_LENGTHS, LINK_MASSES, angles)):
        theta = math.radians(theta_deg)
        x += L * math.cos(theta)
        y += L * math.sin(theta)
        total_pe += m * g * y
    return round(total_pe, 4)

# Random jog commands: (joint_index, delta_degrees)
COMMANDS = []
for _ in range(N_COMMANDS):
    j   = random.randint(0, N_JOINTS - 1)
    d   = random.choice([-1, 1]) * ANGLE_STEP * random.randint(1, 3)
    COMMANDS.append((j, d))

print(f"Robotic arm parameters:")
print(f"  Joints         : {N_JOINTS}")
print(f"  Link lengths   : {LINK_LENGTHS} m")
print(f"  Link masses    : {LINK_MASSES} kg")
print(f"  Jog step       : {ANGLE_STEP}°")
print(f"  Commands       : {N_COMMANDS}")
print(f"  Undos to apply : {N_UNDOS}")
print(f"  Redos to apply : {N_REDOS}")
print()
print("  First 8 commands: (joint, delta°)")
for i, (j, d) in enumerate(COMMANDS[:8]):
    print(f"    [{i+1}] joint {j+1}  {d:+d}°")


## Step 2 — Two-stack Undo / Redo system

In [ ]:
class ArmController:
    """
    Robotic arm controller with undo/redo via two stacks.
    State: list of joint angles (degrees).
    """
    def __init__(self, n_joints):
        self.angles     = [0] * n_joints
        self.undo_stack = []   # each entry: (joint, delta) — what was applied
        self.redo_stack = []   # entries that were undone

    def apply(self, joint, delta):
        """Apply a jog command. Clears redo stack (new branch)."""
        self.angles[joint] += delta
        self.undo_stack.append((joint, delta))
        self.redo_stack.clear()   # new move invalidates redo history

    def undo(self):
        """Reverse the last command."""
        if not self.undo_stack:
            return None
        joint, delta = self.undo_stack.pop()
        self.angles[joint] -= delta          # reverse
        self.redo_stack.append((joint, delta))
        return (joint, -delta)

    def redo(self):
        """Re-apply the last undone command."""
        if not self.redo_stack:
            return None
        joint, delta = self.redo_stack.pop()
        self.angles[joint] += delta
        self.undo_stack.append((joint, delta))
        return (joint, delta)

    def state(self):
        return tuple(self.angles)

ctrl = ArmController(N_JOINTS)
log  = []

# Apply all commands
for j, d in COMMANDS:
    ctrl.apply(j, d)
    log.append(("APPLY", j, d, ctrl.state(), len(ctrl.undo_stack), len(ctrl.redo_stack)))

state_after_all = ctrl.state()

# Apply undos
for _ in range(N_UNDOS):
    result = ctrl.undo()
    if result:
        j, d = result
        log.append(("UNDO", j, d, ctrl.state(), len(ctrl.undo_stack), len(ctrl.redo_stack)))

state_after_undo = ctrl.state()

# Apply redos
for _ in range(N_REDOS):
    result = ctrl.redo()
    if result:
        j, d = result
        log.append(("REDO", j, d, ctrl.state(), len(ctrl.undo_stack), len(ctrl.redo_stack)))

state_final = ctrl.state()

print(f"Command execution log (last 12 entries):")
print(f"  {'Op':<6}  {'J':>2}  {'Δ°':>5}  {'Angles (°)':<35}  {'Undo':>5}  {'Redo':>5}")
print(f"  {'─'*6}  {'─'*2}  {'─'*5}  {'─'*35}  {'─'*5}  {'─'*5}")
for op, j, d, angles, nu, nr in log[-12:]:
    ang_str = str(list(angles))
    print(f"  {op:<6}  {j+1:2d}  {d:+5d}  {ang_str:<35}  {nu:5d}  {nr:5d}")

print(f"\nState after all commands : {list(state_after_all)}")
print(f"State after {N_UNDOS} undos    : {list(state_after_undo)}")
print(f"State after {N_REDOS} redos    : {list(state_final)}")


## Step 3 — Min-stack: O(1) minimum potential energy tracking

In [ ]:
class MinStack:
    """
    Stack that tracks the minimum value in O(1).
    Auxiliary stack stores (current_min) at each push.
    """
    def __init__(self):
        self._main = []   # (value, )
        self._mins = []   # running minimum at each level

    def push(self, val):
        self._main.append(val)
        cur_min = val if not self._mins else min(val, self._mins[-1])
        self._mins.append(cur_min)

    def pop(self):
        if self._main:
            self._mins.pop()
            return self._main.pop()

    def get_min(self):
        return self._mins[-1] if self._mins else None

    def __len__(self): return len(self._main)

# Replay all APPLY commands, track PE at each step
pe_stack = MinStack()
ctrl2    = ArmController(N_JOINTS)
pe_history = []

for j, d in COMMANDS:
    ctrl2.apply(j, d)
    pe = potential_energy(list(ctrl2.state()))
    pe_stack.push(pe)
    pe_history.append(pe)

min_pe        = pe_stack.get_min()
min_pe_idx    = pe_history.index(min_pe)
current_pe    = pe_history[-1]

print(f"Potential energy tracking (min-stack):")
print(f"  PE after all commands  : {current_pe:.4f} J")
print(f"  Minimum PE seen        : {min_pe:.4f} J  (after command {min_pe_idx+1})")
print(f"  Min-stack depth        : {len(pe_stack)}")
print()
print("  PE profile (every 4th command):")
print(f"  {'Cmd':>5}  {'PE (J)':>10}  {'Running min':>12}")
print(f"  {'─'*5}  {'─'*10}  {'─'*12}")
# rebuild running min for display
rmin = float('inf')
for i, pe in enumerate(pe_history):
    rmin = min(rmin, pe)
    if i % 4 == 0 or i == len(pe_history)-1:
        print(f"  {i+1:5d}  {pe:10.4f}  {rmin:12.4f}")


## Step 4 — Balanced configuration check (stack-based symmetry)

In [ ]:
def count_sign_changes(angles):
    """
    Use a stack to count sign changes in the angle sequence
    (a proxy for how many direction reversals the arm makes).
    """
    stack = []
    changes = 0
    for a in angles:
        if a == 0: continue
        sign = 1 if a > 0 else -1
        if stack and stack[-1] != sign:
            changes += 1
        stack.append(sign)
    return changes

# Check the entire trajectory angle values per joint
traj = []
ctrl3 = ArmController(N_JOINTS)
for j, d in COMMANDS:
    ctrl3.apply(j, d)
    traj.append(list(ctrl3.state()))

per_joint_changes = []
for jj in range(N_JOINTS):
    joint_angles = [state[jj] for state in traj]
    per_joint_changes.append(count_sign_changes(joint_angles))

print("Joint trajectory analysis:")
print(f"  {'Joint':>6}  {'Final angle (°)':>16}  {'Sign changes':>13}")
print(f"  {'─'*6}  {'─'*16}  {'─'*13}")
for jj in range(N_JOINTS):
    print(f"  {jj+1:6d}  {list(ctrl3.state())[jj]:16d}  {per_joint_changes[jj]:13d}")

total_sign_changes = sum(per_joint_changes)
print(f"\n  Total direction reversals: {total_sign_changes}")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " ROBOTIC ARM UNDO/REDO — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  ARM PARAMETERS" + " "*(W-16) + "║")
print(f"║  {'Joints':<28}: {N_JOINTS:<26}║")
print(f"║  {'Link lengths':<28}: {LINK_LENGTHS}{'':<10}║"[:W+2] + "║")
print(f"║  {'Commands applied':<28}: {N_COMMANDS:<26}║")
print(f"║  {'Undos / Redos':<28}: {N_UNDOS} / {N_REDOS}{'':<22}║")
print("╠" + "═"*W + "╣")
print("║  UNDO/REDO RESULTS" + " "*(W-19) + "║")
print(f"║  {'State after all commands':<28}: {list(state_after_all)}{'':<5}║"[:W+2] + "║")
print(f"║  {'State after undos':<28}: {list(state_after_undo)}{'':<5}║"[:W+2] + "║")
print(f"║  {'Final state':<28}: {list(state_final)}{'':<5}║"[:W+2] + "║")
print("╠" + "═"*W + "╣")
print("║  MIN-STACK (ENERGY)" + " "*(W-20) + "║")
print(f"║  {'Min PE in trajectory':<28}: {min_pe:.4f} J{'':<20}║")
print(f"║  {'After command #':<28}: {min_pe_idx+1:<26}║")
print(f"║  {'Final PE':<28}: {current_pe:.4f} J{'':<20}║")
print("╠" + "═"*W + "╣")
print("║  TRAJECTORY ANALYSIS" + " "*(W-21) + "║")
print(f"║  {'Total direction reversals':<28}: {total_sign_changes:<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the system?

*Your answer here:*

---

**Q2 — Which stack or queue concept did you find most important, and why?**
Pick one technique from the notebook and explain in your own words what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
